In [ ]:
import pandas
from helpers.data import MinioHelper, CalcHelper
import plotly.express as px

In [ ]:
YEARS = ['2023']
SEASON = 'regular'
BUCKET = 'football-warehouse'

In [ ]:
helper = MinioHelper(BUCKET)

game_stats_frame = helper.get_statistics(YEARS, SEASON, 'games')
stats_frame = helper.get_statistics(YEARS, SEASON, 'teams')

In [ ]:
stats_pivot = stats_frame.pivot_table(index=['team', 'opponent', 'year', 'week'], columns='statistic_name', values='statistic_value', aggfunc='sum', fill_value=0)

In [ ]:
stats_pivot = CalcHelper.calculate_efficiency(stats_pivot)

In [ ]:
stats_flat = stats_pivot.reset_index()

In [ ]:
drop_columns = ['game_id', 'location', 'city', 'state', 'game_date', 'is_conference', 'note', 'line', 'over_under', 'game_type']

home_drops = ['away_team']
home_drops.extend(drop_columns)

away_drops = ['home_team']
away_drops.extend(drop_columns)

home_score_df = game_stats_frame.drop(labels=home_drops, axis=1)
home_score_df = home_score_df.rename(columns={'home_team': 'team', 'home_score': 'points', 'away_score': 'pointsagainst'})

away_score_df = game_stats_frame.drop(labels=away_drops, axis=1)
away_score_df = away_score_df.rename(columns={'away_team': 'team', 'away_score': 'points', 'home_score': 'pointsagainst'})


In [ ]:
score_frame = pandas.concat([home_score_df, away_score_df],ignore_index=True)

In [ ]:
stats_joined_frame = stats_flat.merge(score_frame, left_on=['team', 'year', 'week'], right_on=['team', 'year', 'week'], how='left')

In [ ]:
stats_joined_frame = CalcHelper.calculate_result(stats_joined_frame)
stats_joined_frame.info()

In [ ]:
fig = px.scatter(stats_joined_frame, x='points', y='rushingyards', color='result')
fig.show()